# 対話の履歴を管理して、長くなったら古いターンを捨てる

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/19_dialogue/chat_history.ipynb)

NLP クックブック 19-05 ／ 対話システム

LLM と何ターンも対話を続けたい。履歴が長くなると入力が膨らんで遅くなり、やがて上限を超えるので、長さを管理したい。

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"   # 3B の日本語モデル。CPU でも 1 応答あたり数秒〜十数秒 (メモリ約 7GB)
tokenizer = AutoTokenizer.from_pretrained(name)
llm = AutoModelForCausalLM.from_pretrained(name).eval()

class Chat:
    def __init__(self, system, max_input_tokens=400):
        self.system = {"role": "system", "content": system}
        self.history = []                       # user と assistant の発話を交互に積む
        self.max_input_tokens = max_input_tokens

    def n_tokens(self, messages):
        return len(tokenizer.apply_chat_template(messages, add_generation_prompt=True))

    def window(self):
        """システムプロンプトは必ず残し、上限に収まるまで古いターン (user と assistant の組) を捨てる"""
        history = list(self.history)
        while len(history) > 1 and self.n_tokens([self.system] + history) > self.max_input_tokens:
            history = history[2:]
        return [self.system] + history

    def say(self, text, max_new_tokens=80):
        self.history.append({"role": "user", "content": text})
        messages = self.window()
        inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True)
        with torch.no_grad():
            out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        reply = tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        self.history.append({"role": "assistant", "content": reply})
        return reply, inputs["input_ids"].shape[1], (len(messages) - 1) // 2

chat = Chat("あなたは旅行の相談に乗るアシスタントです。2 文以内で簡潔に答えてください。")
for user in ["私の名前は田中です。来月、京都に 2 泊で行く予定です。", "秋におすすめのお寺を 1 つ教えてください。", "そこへは京都駅からどう行けばいいですか？",
             "近くでお昼を食べるなら何がいいですか？", "ところで、私の名前を覚えていますか？"]:
    reply, n_in, n_turns = chat.say(user)
    print(f"あなた: {user}\nボット: {reply}\n   (入力 {n_in} トークン、直近 {n_turns} ターン分を使用)")

あなた: 私の名前は田中です。来月、京都に 2 泊で行く予定です。
ボット: 田中様、京都へのご旅行楽しみですね！来月の京都では、清水寺や嵐山、金閣寺などの観光スポットを訪れるのはいかがでしょうか。また、京都の食文化もお楽しみください。例えば、湯豆腐や抹茶スイーツなどがおすすめです。素敵な旅になりますように！
   (入力 39 トークン、直近 0 ターン分を使用)
あなた: 秋におすすめのお寺を 1 つ教えてください。
ボット: 秋に京都でおすすめなのは、東福寺です。紅葉の季節には、通天橋から眺める紅葉のトンネルが非常に美しく、多くの観光客が訪れます。ぜひお楽しみください。
   (入力 104 トークン、直近 1 ターン分を使用)
あなた: そこへは京都駅からどう行けばいいですか？
ボット: 東福寺へは、京都駅から徒歩約30分でも可能ですが、市バスを利用するのが便利です。京都駅から「東福寺」または「東福寺前」バス停下車すぐです。バスの系統は208番などがおすすめです。所要時間は約10分ほどです。
   (入力 150 トークン、直近 2 ターン分を使用)
あなた: 近くでお昼を食べるなら何がいいですか？
ボット: 東福寺の近くでお昼を食べるなら、湯豆腐や京懐石がおすすめです。特に、東福寺近くの「六波羅蜜寺」周辺や「鳥羽街道」沿いには、美味しい湯豆腐店や和食のお店が多数あります。例えば、「奥丹」や「南禅寺 順正 東福寺店」などが人気です。また、京うどんやarabesque（アラベスク：arabesqueは
   (入力 217 トークン、直近 3 ターン分を使用)
あなた: ところで、私の名前を覚えていますか？
ボット: いいえ、申し訳ありませんが、私はあなたのお名前を田中様と伺いましたが、会話の中で別のお名前で呼ばれていないか確認できませんでした。もし差し支えなければ、今後のために正式名称を教えていただけると幸いです。

ただし、会話の中では「あなた」や「田中様」といった呼び方で対応させていただくことも可能です。どのようにお呼びすればよいか、お決めいただければと思います。
   (入力 311 トークン、直近 4 ターン分を使用)


- LLM は状態を持たない。「覚えている」ように見えるのは、毎回これまでの対話をすべて入力しているから。
- 古いターンを捨てると、そこで話したこと (最初に名乗った名前) は忘れる。上限を小さくしてあるこの例では、最後の質問でそれが起きる。
- 捨てるときは user と assistant の組で捨てる。片方だけ残すと、チャットテンプレートの交互の並びが崩れる。
- 大事な情報を失わないための方法が、次項の「要約して覚えておく」。